# 08S · T-cell soft gene weighting → GPU PCA → GPU Harmony

**Project:** Adagrasib_mouse_project_VisiumHD_2026  
**Release:** 1.0.1 — live GPU startup diagnostics  
**Question:** Does smooth downweighting of genes enriched in the local non-T environment improve the separation of CD4-associated, CD8-associated, and Treg-associated evidence?

This notebook runs a matched **unweighted** and **ratio-weighted** comparison on **every cell in the completed 08O T cell 1 + T cell 2 analysis**. Both analyses use the same broad gene set, normalization, PCA settings, sample integration, graph construction and clustering resolutions. Their gene weights are the experimental difference.

## Start here

1. Put this notebook in the project folder used for 08O/08Q, or a subfolder of it.
2. Select your working **RAPIDS GPU Python kernel**.
3. Leave the setup cell below unchanged unless the project folder needs to be specified; choose **Run All**.
4. When it finishes, review the inline plots and open the complete HTML report. Send back the small **08S_return_to_chat.zip** file produced in the last step.

The notebook includes its helpers. It discovers the existing `08O_latest_result.json` and prints the exact H5AD and ratio table it will use. **No H5AD picker or analysis-mode selector is needed.** The data remain on your instance.

The first run uses up to **two visible GPUs**, one per matched analysis; all cells participate in each. A short synthetic GPU check runs before the large input is read. Preparation and plotting use the CPU. The GPU stage has a 100-minute budget and saved checkpoints; total time also includes preparation and review, so the 1–2 hour target is not a measured runtime guarantee.

In [1]:
from pathlib import Path

# Usually leave this unchanged. Otherwise give the ORIGINAL project
# folder containing ct26_analysis_config.json and inputs/pipeline_config.json.
PROJECT_DIR = Path.cwd()

# First test: 0.75. A later 0.60 test downweights environmental genes more.
# This is the midpoint of a SMOOTH curve, not a gene-removal cutoff.
WEIGHT_MIDPOINT = 0.75

### Included code

The next cell restores the versioned helper files included inside this notebook. It needs no download and can remain collapsed. Readable copies of the same code are also included in the ZIP package.

In [2]:
# The three readable helper modules are embedded in this notebook.
# This cell uses only Python's standard library to restore them locally.
import base64, hashlib, importlib, json, sys, zlib
_payload = zlib.decompress(base64.b64decode('...'))
if hashlib.sha256(_payload).hexdigest() != '4720c4ae337f012b5073be934d8ad37c804a7ad4966d3d9fa2f3db352aff4a7a':
    raise RuntimeError("The embedded 08S helper bundle is damaged; download the notebook again.")
_sources = json.loads(_payload)
HELPER_DIR = (Path.cwd() / "08S_runtime" / '4720c4ae337f012b').resolve()
HELPER_DIR.mkdir(parents=True, exist_ok=True)
for _name, _source in _sources.items():
    _destination = HELPER_DIR / _name
    if not _destination.is_file() or _destination.read_text() != _source:
        _temporary = _destination.with_suffix(".py.partial")
        _temporary.write_text(_source)
        _temporary.replace(_destination)
# Make repeated runs use the exact embedded version, not another helper
# from an older notebook directory. No CUDA is initialized in this kernel.
sys.path.insert(0, str(HELPER_DIR))
importlib.invalidate_caches()
for _module in ("tcell_weighted_workflow", "tcell_weighted_gpu", "tcell_weighted_review"):
    sys.modules.pop(_module, None)
try:
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    import anndata
    import h5py
    import matplotlib.pyplot as plt
    import tcell_weighted_workflow as workflow
    import tcell_weighted_review as marker_review
    from tcell_weighted_gpu import DEFAULT_SETTINGS
except ImportError as exc:
    raise ImportError("Select the same RAPIDS Python kernel that ran the previous GPU notebook. "
                      "This notebook also needs NumPy, SciPy, pandas, anndata, h5py and matplotlib.") from exc
from IPython.display import display, Markdown, Image, FileLink
GPU_SETTINGS = dict(DEFAULT_SETTINGS)
GPU_SETTINGS["max_wall_minutes"] = 100.0
print("08S notebook release:", workflow.RELEASE_VERSION)
print("Python kernel:", sys.executable)
print("Only PROJECT_DIR needs adjustment if automatic input discovery fails.")

08S notebook release: 1.0.1
Python kernel: /home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python
Only PROJECT_DIR needs adjustment if automatic input discovery fails.


## 1. Locate the completed T-cell result and check the GPU environment

The exact input paths are printed below. Input discovery uses the project configuration or the existing 08O pointer. The GPU check uses 192 synthetic observations solely to verify the installed PCA/Harmony/graph/UMAP APIs; the biological analysis then uses the complete T-cell subset.

**Startup diagnostics in release 1.0.1:** each import and GPU stage now prints its progress immediately. During a quiet stage, a status line appears every 15 seconds. The startup ceiling is 10 minutes to allow for initial imports and compilation; that ceiling is not an expected runtime.

If startup fails or is interrupted, this cell offers **08S_preflight_diagnostics.zip**, containing the full startup log and status report. Send that small ZIP back. The biological preparation stage has not started yet. These reporting changes preserve the weighting, clustering settings and compatible caches.

In [3]:
import time
_setup_started = time.perf_counter()
INPUTS = workflow.resolve_inputs(
    PROJECT_DIR,
    allow_known_fallback=(Path(PROJECT_DIR).expanduser().resolve() == Path.cwd().resolve()),
)
try:
    RUNTIME = workflow.preflight_gpu(HELPER_DIR)
except BaseException as _startup_error:
    _diagnostic_zip = getattr(_startup_error, "diagnostic_zip", None)
    if _diagnostic_zip is not None and Path(_diagnostic_zip).is_file():
        import os
        display(Markdown("### Startup diagnostics saved\nPlease return the small ZIP below so the exact failing stage and traceback can be inspected."))
        display(FileLink(os.path.relpath(_diagnostic_zip, Path.cwd()),
                         result_html_prefix="<strong>Return this diagnostic ZIP:</strong> "))
        print("Diagnostic ZIP:", Path(_diagnostic_zip).resolve())
    raise
print("Visible GPUs:", RUNTIME["visible_device_count"])
print("GPU check:", RUNTIME["arithmetic_check"])
print("RAPIDS version:", RUNTIME["rapids_singlecell_version"])
SETUP_SECONDS = time.perf_counter() - _setup_started

Completed 08O input: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/08_ct26_joint_scvi/CT26_diam50_100_mt10_g50_c100_scvi4k_latent30_seed17/14_tcell_ratio_subclustering/02_joint_tcell_analysis/5f02bd423214b860943d4edc/analysis.h5ad
Ratio audit: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/08_ct26_joint_scvi/CT26_diam50_100_mt10_g50_c100_scvi4k_latent30_seed17/14_tcell_ratio_subclustering/02_joint_tcell_analysis/5f02bd423214b860943d4edc/ratio_filter_in_analysis.csv
New 08S output: /stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor/derived_files/tbio8110_stardist_proseg_resolvi_v1/08_ct26_joint_scvi/CT26_diam50_100_mt10_g50_c100_scvi4k_latent30_seed17/18_tcell_soft_weight_harmony
GPU startup check: live output below; timeout 600 seconds.
Startup log: /home/domino/reny28/Projects/Visium_projects/TBIO-8110/code/python/08S_Tcell_SoftWeight_Harmony_GPU1.1/0

### Startup diagnostics saved
Please return the small ZIP below so the exact failing stage and traceback can be inspected.

/home/domino/reny28/Projects/Visium_projects/TBIO-8110/code/python/08S_Tcell_SoftWeight_Harmony_GPU1.1/08S_diagnostics/preflight_20261110T165311Z_3c74a90a/08S_preflight_diagnostics.zip

Diagnostic ZIP: /home/domino/reny28/Projects/Visium_projects/TBIO-8110/code/python/08S_Tcell_SoftWeight_Harmony_GPU1.1/08S_diagnostics/preflight_20261110T165311Z_3c74a90a/08S_preflight_diagnostics.zip


GPUPreflightError: GPU startup check exited with status 1. The captured stdout/stderr and any traceback are saved. Return this diagnostic ZIP: /home/domino/reny28/Projects/Visium_projects/TBIO-8110/code/python/08S_Tcell_SoftWeight_Harmony_GPU1.1/08S_diagnostics/preflight_20261110T165311Z_3c74a90a/08S_preflight_diagnostics.zip

## 2. Prepare one shared, broad gene matrix

For original count $c_{ig}$ and full-source library size $L_i$, this test uses

$$
x_{ig} = \log\left(1+10^4 c_{ig}/L_i\right), \qquad
w_g = \frac{1}{1+(r_g/0.75)^2}.
$$

Here $r_g$ is the existing **08O pooled neighbor-to-T expression ratio**. Smaller ratios receive more influence. The selected midpoint replaces 0.75 in the equation when you change the setup value.

The weighted PCA centers the columns of $x_{ig}\sqrt{w_g}$; the control centers $x_{ig}$. Thus $w_g$ is a gene's relative contribution to squared expression distance before PCA. **There is no gene-wise unit-variance scaling after weighting**, which would cancel a fixed weight.

Both branches restore eligible genes from the full common-gene universe in 08O. The only feature filters are detection in at least 20 cells and nonzero normalized variance. Previous HVG and ratio-pass masks are ignored. Every 08O T cell remains in the analysis, including any cells without an observed eligible feature; their number is explicitly reported.

These are fixed **feature-importance weights**, derived from the existing pooled spatial-ratio analysis. They are applied to every T cell. This pilot does not estimate a cell-specific endogenous transcript fraction. Full-source library totals are used before the weighting, and original counts are retained for marker review.

In [ ]:
_prepare_started = time.perf_counter()
CACHE_DIR, CACHE = workflow.prepare_cache(
    INPUTS,
    marker_review.ALL_MARKERS,
    midpoint=float(WEIGHT_MIDPOINT),
    power=2.0,
    min_detected_cells=20,
    target_sum=10000.0,
    resume=True,
)
PREPARE_SECONDS = time.perf_counter() - _prepare_started
display(pd.DataFrame([{
    "T cells analyzed": CACHE["n_cells"],
    "samples": CACHE["n_samples"],
    "conditions": CACHE["n_conditions"],
    "source genes": CACHE["n_source_genes"],
    "eligible shared genes": CACHE["n_features"],
    "zero eligible-feature cells retained": CACHE["zero_feature_cells"],
    "sparse feature matrix GiB": round(CACHE["csr_bytes"] / 2**30, 3),
    "preparation minutes this invocation": round(PREPARE_SECONDS / 60, 2),
}]))
FEATURE_TABLE = pd.read_csv(CACHE_DIR / "features.csv")
GENE_AUDIT = pd.read_csv(CACHE_DIR / "feature_audit_all_genes.csv")
_check_genes = ["Cd3e", "Trac", "Cd4", "Cd8a", "Cd8b1", "Foxp3", "Il2ra",
                "Ctla4", "Tcf7", "Gzmb", "Mki67", "Lyz2", "Epcam", "Col1a1"]
display(GENE_AUDIT.set_index("gene").reindex(_check_genes)[
    ["ratio", "detected_cells_08S", "eligible_08S", "weight_08S", "eligibility_reason"]
])
print("Unavailable marker symbols:", ", ".join(CACHE["missing_marker_genes"]) or "None")

In [ ]:
# Visual audit of the chosen smooth weighting curve and retained genes.
_ratios = FEATURE_TABLE["ratio"].to_numpy(dtype=float)
_upper = max(2.0, min(10.0, float(np.quantile(_ratios, 0.99))))
_grid = np.linspace(0, _upper, 500)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].plot(_grid, workflow.soft_weights(_grid, WEIGHT_MIDPOINT, 2), color="#166b85", lw=2.5)
axes[0].axvline(WEIGHT_MIDPOINT, color="#9ba5af", ls="--", lw=1)
axes[0].set(xlabel="Existing neighbor-to-T ratio", ylabel="Distance weight",
            title="Smooth feature weighting", ylim=(0, 1.03))
axes[1].hist(FEATURE_TABLE["weight"], bins=40, color="#527e9e", edgecolor="white")
axes[1].set(xlabel="Distance weight", ylabel="Number of eligible genes",
            title=f"{len(FEATURE_TABLE):,} genes used in both analyses")
fig.tight_layout()
plt.show()

## 3. Run GPU PCA, Harmony, neighbors, clustering and UMAP

| Setting | First-run value |
|---|---|
| PCA | 50 centered components; sparse randomized GPU computation |
| PCA accuracy settings | 20 extra dimensions; 3 power iterations |
| Harmony | Joint PCA; integrate by `sample_id`; theta 1.0 |
| Harmony implementation | Original Harmony flavor, explicitly selected when the installed API exposes a flavor |
| Harmony work limit | 100 internal soft clusters; 10 outer / 20 inner iterations |
| Neighbor graph | 30 neighbors; GPU CAGRA; Euclidean distance |
| Leiden resolutions | 0.2, 0.4, 0.6 |
| UMAP | GPU; 200 iterations; minimum distance 0.3 |
| Seed | 0 for both analyses |

A **single joint PCA basis per branch** is fitted across all samples. Harmony then adjusts that shared representation by sample. Condition is retained for biological review and is not passed as a Harmony covariate. Separately fitted sample-specific PC axes would not provide the shared coordinate system used here.

The algorithms use all cells. The computation keeps expression sparse and forms only thin dense PCA work arrays. With two or more visible GPUs, the unweighted and weighted analyses run concurrently in separate processes; with one visible GPU they run sequentially. The four cards' memory is not pooled. PCA, Harmony, graph and individual Leiden results are checkpointed.

In [ ]:
_gpu_started = time.perf_counter()
BRANCHES, RUN_DIR = workflow.launch_comparison(
    CACHE_DIR,
    HELPER_DIR,
    RUNTIME,
    settings=GPU_SETTINGS,
    max_minutes=100.0,
    resume=True,
)
GPU_SECONDS = time.perf_counter() - _gpu_started
print("Matched comparison completed in", round(GPU_SECONDS / 60, 2), "minutes this invocation.")
for _branch, _path in BRANCHES.items():
    print(_branch + ":", _path)
print("Comparison folder:", RUN_DIR)

## 4. Review clusters using original marker expression

**Dot size:** fraction of cells with original observed count > 0.  
**Dot color:** mean original, unweighted log1p expression, including zeros.  
Both methods share the same color limits and marker statistics use **every cell**. Only the UMAP display is limited to 120,000 identical, reproducibly selected cells per method.

Core lineage dotplots are produced at all three resolutions; T-cell state and environmental-marker panels use resolution 0.4. Every cluster is included, with multiple pages if necessary. A supplementary gene-scaled panel makes low-expression patterns easier to compare; its color scale is explicitly labeled.

| Review panel | Examples |
|---|---|
| T-lineage support | Cd3d, Cd3e, Trac, Trbc1, Trbc2 |
| CD4 / CD8 transcripts | Cd4; Cd8a, Cd8b1 |
| Treg-associated program | Foxp3, Il2ra, Ctla4, Ikzf2, Tnfrsf18, Icos |
| Naive / memory | Tcf7, Lef1, Ccr7, Sell, Il7r, Ltb |
| Cytotoxic | Gzmb, Gzmk, Prf1, Ccl5, Nkg7 |
| Activation / dysfunction | Pdcd1, Tox, Lag3, Havcr2, Tigit, Cd69 |
| Cycling | Mki67, Top2a, Stmn1 |
| NK / environmental review | Ncr1, Klrd1, Lyz2, Epcam, Krt8/18, Col1a1/2 |

These marker programs guide inspection; they do not force cell labels. Naive/memory and cytotoxic genes are not exclusive to one T lineage. A Treg-like group is more persuasive when Foxp3 is supported by additional Treg-associated and T-lineage evidence. Missing transcripts in sparse data remain uncertain.

In [ ]:
_review_started = time.perf_counter()
LOG_MARKERS, COUNT_MARKERS, MARKER_GENES, OBS, METHODS = workflow.load_review_data(CACHE_DIR, BRANCHES)
REVIEW_DIR = RUN_DIR / "review"
REVIEW = marker_review.build_review(
    LOG_MARKERS, COUNT_MARKERS, MARKER_GENES, OBS, METHODS, REVIEW_DIR,
    sample_key="sample_id", condition_key="condition", depth_key="total_counts",
    cluster_keys=[f"leiden_{r:g}" for r in GPU_SETTINGS["resolutions"]],
    primary_cluster_key="leiden_0.4",
    max_plot_cells=120000,
    max_clusters_per_page=35,
    random_state=0,
    normalization_target_sum=10000.0,
)
REVIEW_SECONDS = time.perf_counter() - _review_started
COMPARISON = pd.read_csv(REVIEW_DIR / "method_comparison.csv")
display(COMPARISON)
print("Complete HTML review:", REVIEW_DIR / "review.html")
print("All marker statistics:", REVIEW_DIR / "marker_statistics.csv.gz")
print("All cluster summaries:", REVIEW_DIR / "cluster_summary.csv")

### Main plots

The notebook displays the UMAP overview, sample/condition/depth review and primary lineage dotplots below. The HTML report includes all resolutions, state/background dotplots, gene expression maps, condition-stratified UMAPs and the supporting tables. The axes within each condition series use the same joint UMAP coordinates.

In [ ]:
for _filename in ("umap_clusters.png", "umap_sample_condition_depth.png", "umap_lineage_markers.png"):
    display(Image(filename=str(REVIEW_DIR / "figures" / _filename), width=1150))
for _method in ("unweighted", "weighted"):
    _dotplots = sorted((REVIEW_DIR / "figures").glob(
        f"dotplot_{_method}_leiden_0_4_lineage_mean_log1p_page*.png"))
    for _path in _dotplots[:2]:
        display(Image(filename=str(_path), width=1100))
    if len(_dotplots) > 2:
        print(f"{_method}: {len(_dotplots)} lineage pages in total; all are in the HTML review.")

## 5. Collect results and create the compact return file

The final ZIP contains aggregate marker/cluster/sample/depth diagnostics, feature weights, timing metadata and selected figures. It excludes the large expression matrices and embedding H5ADs. Its selected input files are capped at approximately 25 MB before ZIP compression; any omitted figures are listed explicitly in `PACKAGE_CONTENTS.json`. The full report remains in the analysis output folder.

A convenient local copy of the return ZIP is placed in `08S_results/<run ID>/` beneath the notebook's working folder. The full HTML report and its figures are also copied there so its links can be opened together. The original result paths and the local download paths are printed.

In [ ]:
import os, shutil
TIMINGS = {
    "setup_seconds": SETUP_SECONDS,
    "preparation_seconds_this_invocation": PREPARE_SECONDS,
    "comparison_seconds_this_invocation": GPU_SECONDS,
    "review_seconds_this_invocation": REVIEW_SECONDS,
    "note": "Two GPU branches can overlap; cached work may have run in an earlier invocation.",
}
workflow.save_json(REVIEW_DIR / "notebook_timings.json", TIMINGS)
RETURN_ZIP = workflow.make_return_package(RUN_DIR, CACHE_DIR, REVIEW_DIR, max_megabytes=25)
LOCAL_RESULTS = Path.cwd() / "08S_results" / (CACHE["cache_id"][:8] + "_" + RUN_DIR.name)
LOCAL_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy2(RETURN_ZIP, LOCAL_RESULTS / RETURN_ZIP.name)
shutil.copytree(REVIEW_DIR, LOCAL_RESULTS / "review", dirs_exist_ok=True)
display(Markdown("### Ready to inspect"))
display(FileLink(os.path.relpath(LOCAL_RESULTS / "review/review.html", Path.cwd()),
                 result_html_prefix="Complete report: "))
display(FileLink(os.path.relpath(LOCAL_RESULTS / RETURN_ZIP.name, Path.cwd()),
                 result_html_prefix="<strong>Return this ZIP:</strong> "))
print("Local report:", (LOCAL_RESULTS / "review/review.html").resolve())
print("Local return ZIP:", (LOCAL_RESULTS / RETURN_ZIP.name).resolve())
display(pd.DataFrame([{key: round(value / 60, 2) for key, value in TIMINGS.items()
                       if key.endswith("seconds") or key.endswith("invocation")}]).rename(
    columns=lambda name: name.replace("_seconds", "_minutes")))

## How we will decide whether this helped

Compare the methods at the **same resolution** and inspect support across samples and depth strata:

1. **Lineage structure:** Cd4 and Cd8a/Cd8b1 become enriched in different groups with T-lineage support, instead of all lineage transcripts concentrating in a single high-count group.
2. **Treg evidence:** Foxp3 and supporting genes show a coherent pattern beyond a few isolated transcript-positive cells. Ctla4 or Il2ra alone is insufficient to establish a Treg identity.
3. **Technical structure:** check the relationship of clusters with full-source counts, detected genes, sample and condition. Sample mixing is contextual evidence; stronger mixing alone does not establish better biology.
4. **Fragmentation:** inspect cluster counts, small-cluster cell fractions and whether similar marker programs are split among many tiny groups.
5. **Consistency:** use the by-sample and within-sample depth-quintile marker tables to distinguish a reproducible pattern from one driven by a sample or count stratum.

`method_comparison.csv` includes descriptive log-depth association, sample/condition association and concordance with observed multi-gene evidence. The observed evidence groups cover only the cells with measured supporting combinations; they are **not reference labels**, and higher concordance is not an independent accuracy estimate. Some of these marker genes also contribute to the embedding. More fragmented clustering can mechanically increase some concordance measures.

**A later weight test:** change `WEIGHT_MIDPOINT` from `0.75` to `0.60` and Run All. Compatible normalized arrays and the unweighted control are reused. A lower midpoint applies stronger downweighting. Changing other numerical settings can create a new comparison folder. Interrupted stages can be resumed by rerunning with the same settings; completed checkpoints survive.

**What this establishes for the larger project:** this first experiment evaluates the existing pooled T-compartment weights in a shared PCA space. A useful result would justify developing sample-aware, lineage-specific weighting for the wider dataset. The T-centric vector in this pilot is specific to the T-cell comparison; an all-lineage extension needs weights defined for its corresponding compartments and a shared integration strategy.

## Methods and validation

GPU PCA uses a sparse, implicitly centered randomized range finder with reorthogonalization and a small SVD, following the standard randomized matrix-decomposition framework. Substantial linear algebra runs on the GPU; no dense cell-by-gene or full gene-covariance matrix is constructed. Harmony is run on the resulting shared PCs before the graph, using the RAPIDS implementation. Installed package versions and actual call arguments are recorded.

The delivery was checked with synthetic input/serialization tests and with the same sparse PCA operator compared against a directly centered dense SVD on small CPU examples. Actual CUDA execution, biological outcomes and runtime on your instance could not be tested in the build environment. The first notebook check exercises the small complete GPU path on your installed environment.

References:

- [Halko, Martinsson and Tropp: randomized matrix decompositions](https://arxiv.org/abs/0909.4061).
- [RAPIDS single-cell Harmony integration](https://rapids-singlecell.scverse.org/en/stable/api/generated/rapids_singlecell.pp.harmony_integrate.html).
- [RAPIDS single-cell neighbors](https://rapids-singlecell.scverse.org/en/stable/api/generated/rapids_singlecell.pp.neighbors.html).
- [SAM project: the original inspiration for feature-weighted manifold analysis](https://github.com/atarashansky/self-assembling-manifold).
- Marker-program sources are linked in the full HTML review and readable review module.